
# Construcción de manifests del corpus

Este notebook integra las anotaciones y metadatos de los conjuntos de datos utilizados en el proyecto y construye los dos manifests base:

- `master_manifest.csv`: csv de eventos acústicos
- `recordings_manifest.csv`: scv que muestra grabaciones disponibles

Las grabaciones utilizadas por el proyecto se almacenan en Google Drive bajo `audio_32khz/`. La adquisición de los archivos de audio se gestiona de manera independiente; este notebook se encarga de integrar los metadatos, verificar los archivos disponibles y construir los manifests utilizados por las etapas posteriores del proyecto.

Fuentes utilizadas:

- PteroSet
- Coffee Farms in Colombia and Costa Rica
- Southwestern Amazon Basin
- WABAD, filtrado a las grabaciones realizadas en Colombia


In [46]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import soundfile as sf
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuración

In [47]:
BASE_DIR = Path("/content/drive/MyDrive/DATASET_PROYECTO")

METADATA_DIR = BASE_DIR / "metadata"
SOURCE_DIR = BASE_DIR / "source_metadata"
AUDIO_DIR = BASE_DIR / "audio_32khz"

MASTER_PATH = METADATA_DIR / "master_manifest.csv"
RECORDINGS_PATH = METADATA_DIR / "recordings_manifest.csv"

SOURCE_DIR.mkdir(parents=True, exist_ok=True)

# carpetas posibles para audios fuente/originales.
RAW_CANDIDATES = [BASE_DIR / "raw_audio", BASE_DIR / "raw_audios", BASE_DIR / "audio_raw",]

RAW_AUDIO_DIR = next((path for path in RAW_CANDIDATES if path.exists()),None,)

print("Proyecto:", BASE_DIR)
print("Audios normalizados:", AUDIO_DIR)
print("Audios fuente:", RAW_AUDIO_DIR)


Proyecto: /content/drive/MyDrive/DATASET_PROYECTO
Audios normalizados: /content/drive/MyDrive/DATASET_PROYECTO/audio_32khz
Audios fuente: /content/drive/MyDrive/DATASET_PROYECTO/raw_audio



## 2. Obtención de anotaciones y metadatos

Se descargan los archivos de anotaciones, taxonomía y metadatos publicados por las fuentes originales. Estos archivos son los que permiten reconstruir los eventos, sus límites tiempo-frecuencia y su identificación taxonómica.


In [48]:
PTERO_DIR = SOURCE_DIR / "PteroSet"
COLCR_DIR = SOURCE_DIR / "Col-CR"
PERU_DIR = SOURCE_DIR / "Peru"
WABAD_DIR = SOURCE_DIR / "WABAD-CO"

SOURCE_FILES = {
    "PteroSet annotations": PTERO_DIR / "annotations_identification.json",
    "PteroSet species annotations": PTERO_DIR / "annotations_species.json",
    "PteroSet species": PTERO_DIR / "species.csv",
    "Col-CR annotations": COLCR_DIR / "annotations.csv",
    "Col-CR species": COLCR_DIR / "species.csv",
    "Peru annotations": PERU_DIR / "annotations.csv",
    "Peru species": PERU_DIR / "species.csv",
    "WABAD annotations": WABAD_DIR / "Pooled annotations.csv",
    "WABAD metadata": WABAD_DIR / "Metadata.csv",
}

for name, path in SOURCE_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"No se encontró {name}: {path}")

    print(f"OK: {name}")

print("\nTodos los archivos fuente están disponibles.")

OK: PteroSet annotations
OK: PteroSet species annotations
OK: PteroSet species
OK: Col-CR annotations
OK: Col-CR species
OK: Peru annotations
OK: Peru species
OK: WABAD annotations
OK: WABAD metadata

Todos los archivos fuente están disponibles.


## 3. Funciones auxiliares

In [49]:
TAXON_FIXES = {
    "melanerpes cruentats": "melanerpes cruentatus",
    "megarynchus pitagua": "megarynchus pitangua",
    "mesembrinibis cayanensis": "mesembrinibis cayennensis",
}

BASE_TARGET_CODES = {
    "atalotriccus pilaris": "ATAPIL",
    "attila spadiceus": "ATTSPA",
    "cacicus cela": "CACCEL",
    "columbina squammata": "COLSQU",
    "crypturellus cinereus": "CRYCIN",
    "crypturellus undulatus": "CRYUND",
    "cyanocorax violaceus": "CYAVIO",
    "leptotila verreauxi": "LEPVER",
    "nyctidromus albicollis": "NYCALB",
    "pachyramphus polychopterus": "PACPOL",
    "patagioenas cayennensis": "PATCAY",
    "pitangus sulphuratus": "PITSUL",
    "psarocolius angustifrons": "PSAANG",
    "ramphastos tucanus": "RHATUC",
    "ramphocelus carbo": "RAMCAR",
    "rupornis magnirostris": "RUPMAG",
    "thraupis palmarum": "THRPAL",
    "tolmomyias flaviventris": "TOLFLA",
    "troglodytes aedon": "TROAED",
    "tyrannus melancholicus": "TYRMEL",
}

def pick_col(df, candidates):
    #devuelve el primer nombre de columna disponible
    for column in candidates:
        if column in df.columns:
            return column

    raise KeyError(
        f"No se encontro ninguna de {candidates} "
        f"Columnas disponibles: {list(df.columns)}"
    )


def normalize_species(series):
    #Normalizacion minima y correcciones taxonomicas conocidas
    return (series.astype("string").str.normalize("NFKC").str.strip().str.casefold().str.replace(r"\s+", " ", regex=True).replace(TAXON_FIXES).replace(
        {
            "????": pd.NA,
            "unknown": pd.NA,
            "unidentified": pd.NA,
            "": pd.NA,
        })
    )


def finish_events(df):
    #Añade las columnas comunes del master manifes
    df = df.copy()

    df["species_scientific"] = normalize_species(df["species_scientific"])
    df["identified"] = df["species_scientific"].notna()
    df["t_min"] = pd.to_numeric(df["t_min"],errors="coerce",)
    df["t_max"] = pd.to_numeric(df["t_max"],errors="coerce",)
    df["f_min_original"] = pd.to_numeric(df["f_min_original"],errors="coerce",)
    df["f_max_original"] = pd.to_numeric(df["f_max_original"],errors="coerce",)

    # el corpus utilizado por el proyecto esta normalizado a 32 kHz.
    #por Nyquist, el rango representable llega hasta 16 kHz
    df["f_min"] = df["f_min_original"].clip(lower=0,upper=16000,)
    df["f_max"] = df["f_max_original"].clip(lower=0,upper=16000,)

    df["use_detector"] = (
        df["t_min"].notna()
        & df["t_max"].notna()
        & df["f_min"].notna()
        & df["f_max"].notna()
        & (df["t_min"] >= 0)
        & (df["f_min"] >= 0)
        & (df["t_max"] > df["t_min"])
        & (df["f_max"] > df["f_min"])
    )

    df["use_classifier"] = (df["use_detector"] & df["identified"] )

    df["is_target"] = (df["species_scientific"].isin(BASE_TARGET_CODES) )

    df["class_label"] = pd.NA

    other_mask = (df["identified"] & ~df["is_target"] )

    df.loc[other_mask,"class_label",] = "OTHER_SPECIES"

    df.loc[df["is_target"],"class_label",] = (df.loc[df["is_target"], "species_scientific",].map(BASE_TARGET_CODES))

    df["recording_uid"] = (df["dataset"] + "::" + df["recording"])

    return df


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    #Ssha256 del archivo normalizado
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

## 4. PteroSet

In [50]:
with open(PTERO_DIR / "annotations_identification.json", encoding="utf-8",) as file:
    ptero_ident_json = json.load(file)

with open(PTERO_DIR / "annotations_species.json", encoding="utf-8",) as file:
    ptero_species_json = json.load(file)

ptero_species_table = pd.read_csv(PTERO_DIR / "species.csv")
ident_annotations = pd.DataFrame(ptero_ident_json["annotations"])
species_annotations = pd.DataFrame(ptero_species_json["annotations"])
sounds = pd.DataFrame(ptero_ident_json["sounds"])
categories = pd.DataFrame(ptero_species_json["categories"])

# Informacon de cada grabacion
sound_info = (sounds.set_index("id").to_dict("index"))

# category_id -> codigo taxonomico
category_code = (categories.set_index("id")["name"].to_dict())

# codigo taxonomico -> nombre cientifico
species_code_col = pick_col(ptero_species_table, ["code", "species_code", "eBird Code", "ebird_code",],)

species_name_col = pick_col(ptero_species_table, ["species", "scientific_name", "Scientific Name",],)

code_to_species = dict(zip(ptero_species_table[species_code_col], ptero_species_table[species_name_col],))

#Las anotaciones de especie son un subconjunto de las anotacione de identificaciom. Si ambos JSON comparten un id se utiliza
# de lo contrario se emparejan por su geometría tiempo-frecuencia

species_by_annotation_id = {}

has_annotation_id = (
    "id" in species_annotations.columns
    and "id" in ident_annotations.columns
)

if has_annotation_id:
    for _, annotation in species_annotations.iterrows():
        species_code = category_code.get(annotation["category_id"])

        species_by_annotation_id[annotation["id"]] = (
            code_to_species.get(species_code, species_code)
        )

    shared_annotations = (
        ident_annotations["id"]
        .isin(species_by_annotation_id)
        .sum()
    )

    use_geometry = (
        shared_annotations != len(species_annotations)
    )

else:
    use_geometry = True

print("Emparejamiento por geometría:", use_geometry)

#en caso de que los dos JSON no compartan el mismo id, se emparejan mediante la geometria de la anotacion
species_by_geometry = {}

if use_geometry:
    for _, annotation in species_annotations.iterrows():
        species_code = category_code.get(annotation["category_id"])

        species_name = code_to_species.get(species_code,species_code,)
        key = (annotation["sound_id"], annotation["t_min"], annotation["t_max"], annotation["f_min"], annotation["f_max"],)
        species_by_geometry[key] = species_name

rows = []

for i, annotation in ident_annotations.iterrows():
    sound = sound_info[annotation["sound_id"]]

    sound_path = sound[pick_col(sounds, ["file_name_path", "file_name", "filename", "path",],)]

    recording = Path(str(sound_path)).name

    project = sound.get("project")

    if not project:
        path_text = str(sound_path)

        project = next(value for value in ["MAP1", "PPA1", "PPA2", "PPA3", "PPA4"] if value in path_text )

    if use_geometry:
        key = (annotation["sound_id"],annotation["t_min"],annotation["t_max"],annotation["f_min"],annotation["f_max"],)

        species_name = species_by_geometry.get(key,pd.NA,)
    else:
        species_name = species_by_annotation_id.get(annotation["id"],pd.NA,)

    rows.append({
        "event_id": f"PteroSet::{i}",
        "dataset": "PteroSet",
        "recording": recording,
        "source_group": project,
        "source_archive": f"{project}.zip",
        "country": "Colombia",
        "species_scientific": species_name,
        "t_min": annotation["t_min"],
        "t_max": annotation["t_max"],
        "f_min_original": annotation["f_min"],
        "f_max_original": annotation["f_max"],
    })

pteroset = finish_events(pd.DataFrame(rows))

assert len(pteroset) == 15372
assert pteroset["recording"].nunique() == 563
assert pteroset["identified"].sum() == 6702

print("Eventos:", len(pteroset))
print("Grabaciones:", pteroset["recording"].nunique())
print("Eventos identificados:", pteroset["identified"].sum())


Emparejamiento por geometría: True
Eventos: 15372
Grabaciones: 563
Eventos identificados: 6702


## 5. Coffee Farms in Colombia and Costa Rica

In [51]:
colcr_annotations = pd.read_csv(COLCR_DIR / "annotations.csv")
colcr_species = pd.read_csv(COLCR_DIR / "species.csv")
file_col = pick_col(colcr_annotations,["Filename", "filename"],)
annotation_code_col = pick_col(colcr_annotations,["Species eBird Code", "species_code", "Species Code",],)
lookup_code_col = pick_col(colcr_species,["Species eBird Code","species_code","Code","code",],)
lookup_name_col = pick_col(colcr_species,["Scientific Name","scientific_name","Species","species",],)

code_to_species = dict(zip(colcr_species[lookup_code_col],colcr_species[lookup_name_col],))

colcr = pd.DataFrame({
    "event_id": ("Col-CR::"+ colcr_annotations.index.astype(str)),
    "dataset": "Col-CR",
    "recording": colcr_annotations[file_col].astype(str),
    "species_scientific": (colcr_annotations[annotation_code_col].map(code_to_species)),
    "t_min": colcr_annotations[pick_col(colcr_annotations,["Start Time (s)", "start_time"],)],
    "t_max": colcr_annotations[pick_col(colcr_annotations,["End Time (s)", "end_time"],)],
    "f_min_original": colcr_annotations[pick_col(colcr_annotations,["Low Freq (Hz)", "low_freq"],)],
    "f_max_original": colcr_annotations[pick_col(colcr_annotations,["High Freq (Hz)", "high_freq"],)],
})

colcr["source_group"] = (colcr["recording"].str.extract(r"_(S\d+)_",expand=False,))
colcr["source_archive"] = "soundscape_data.zip"
colcr["country"] = (colcr["source_group"].map({"S01": "Colombia","S02": "Costa Rica",}))
colcr = finish_events(colcr)

assert len(colcr) == 7338
assert colcr["recording"].nunique() == 34
assert colcr["identified"].sum() == 6952

print("Eventos:", len(colcr))
print("Grabaciones:", colcr["recording"].nunique())
print("Eventos identificados:", colcr["identified"].sum())


Eventos: 7338
Grabaciones: 34
Eventos identificados: 6952


## 6. Southwestern Amazon Basin

In [52]:

peru_annotations = pd.read_csv(
    PERU_DIR / "annotations.csv"
)

peru_species = pd.read_csv(
    PERU_DIR / "species.csv"
)

file_col = pick_col(
    peru_annotations,
    ["Filename", "filename"],
)

annotation_code_col = pick_col(
    peru_annotations,
    [
        "Species eBird Code",
        "species_code",
        "Species Code",
    ],
)

lookup_code_col = pick_col(
    peru_species,
    [
        "Species eBird Code",
        "species_code",
        "Code",
        "code",
    ],
)

lookup_name_col = pick_col(
    peru_species,
    [
        "Scientific Name",
        "scientific_name",
        "Species",
        "species",
    ],
)

code_to_species = dict(
    zip(
        peru_species[lookup_code_col],
        peru_species[lookup_name_col],
    )
)

peru = pd.DataFrame({
    "event_id": (
        "Peru::"
        + peru_annotations.index.astype(str)
    ),
    "dataset": "Peru",
    "recording": peru_annotations[file_col].astype(str),
    "country": "Peru",
    "species_scientific": (
        peru_annotations[annotation_code_col]
        .map(code_to_species)
    ),
    "t_min": peru_annotations[
        pick_col(
            peru_annotations,
            ["Start Time (s)", "start_time"],
        )
    ],
    "t_max": peru_annotations[
        pick_col(
            peru_annotations,
            ["End Time (s)", "end_time"],
        )
    ],
    "f_min_original": peru_annotations[
        pick_col(
            peru_annotations,
            ["Low Freq (Hz)", "low_freq"],
        )
    ],
    "f_max_original": peru_annotations[
        pick_col(
            peru_annotations,
            ["High Freq (Hz)", "high_freq"],
        )
    ],
})

peru["source_group"] = (
    peru["recording"]
    .str.extract(
        r"_(S\d+)_",
        expand=False,
    )
)

peru["source_archive"] = "soundscape_data.zip"

peru = finish_events(peru)

assert len(peru) == 16482
assert peru["recording"].nunique() == 21
assert peru["identified"].sum() == 14798

print("Eventos:", len(peru))
print("Grabaciones:", peru["recording"].nunique())
print("Eventos identificados:", peru["identified"].sum())


Eventos: 16482
Grabaciones: 21
Eventos identificados: 14798



## 7. WABAD-CO

WABAD contiene grabaciones procedentes de múltiples regiones. Para este proyecto se seleccionan únicamente las anotaciones cuyo lugar de grabación corresponde a Colombia.


In [53]:
wabad_annotations = pd.read_csv(WABAD_DIR / "Pooled annotations.csv")

CO_SITES = {"BRE", "CARI", "SAL", "SPMCO", "TAM", "UNI", "VER"}

# Seleccionar únicamente los sitios de Colombia
wabad_co_source = wabad_annotations[
    wabad_annotations["Site"].isin(CO_SITES)
].copy()

wabad_co = pd.DataFrame({
    "event_id": "WABAD-CO::" + wabad_co_source.index.astype(str),
    "dataset": "WABAD-CO",
    "recording": wabad_co_source["Recording"].astype(str).map(lambda x: Path(x).name),
    "source_group": wabad_co_source["Site"].astype(str),
    "source_archive": wabad_co_source["Site"].astype(str) + ".zip",
    "country": "Colombia",
    "species_scientific": wabad_co_source["Species"],
    "t_min": pd.to_numeric(wabad_co_source["Begin_Time_(s)"], errors="coerce"),
    "t_max": pd.to_numeric(wabad_co_source["End_Time_(s)"], errors="coerce"),
    "f_min_original": pd.to_numeric(wabad_co_source["Low_Freq_(Hz)"], errors="coerce"),
    "f_max_original": pd.to_numeric(wabad_co_source["High_Freq_(Hz)"], errors="coerce"),
})

wabad_co = finish_events(wabad_co)

assert len(wabad_co) == 9273
assert wabad_co["recording"].nunique() == 393
assert set(wabad_co["source_group"].unique()) == CO_SITES

print("Eventos:", len(wabad_co))
print("Grabaciones:", wabad_co["recording"].nunique())
print("Sitios:", sorted(wabad_co["source_group"].unique()))

Eventos: 9273
Grabaciones: 393
Sitios: ['BRE', 'CARI', 'SAL', 'SPMCO', 'TAM', 'UNI', 'VER']


## 8. Integración del `master_manifest`

In [54]:
master = pd.concat(
    [
        pteroset,
        colcr,
        peru,
        wabad_co,
    ],
    ignore_index=True,
)

AUDIO_FOLDERS = {
    "PteroSet": AUDIO_DIR / "pteroset",
    "Col-CR": AUDIO_DIR / "colcr",
    "Peru": AUDIO_DIR / "peru",
    "WABAD-CO": AUDIO_DIR / "wabad_co",
}

def normalized_audio_path(dataset, recording):
    return (
        AUDIO_FOLDERS[dataset]
        / f"{Path(recording).stem}.flac"
    )

master["audio_available"] = [
    normalized_audio_path(
        dataset,
        recording,
    ).exists()
    for dataset, recording in zip(
        master["dataset"],
        master["recording"],
    )
]

MASTER_COLUMNS = [
    "event_id",
    "dataset",
    "recording",
    "source_group",
    "source_archive",
    "country",
    "species_scientific",
    "identified",
    "t_min",
    "t_max",
    "f_min",
    "f_max",
    "is_target",
    "class_label",
    "f_min_original",
    "f_max_original",
    "use_detector",
    "use_classifier",
    "recording_uid",
    "audio_available",
]

master = master[MASTER_COLUMNS]

display(
    master.groupby("dataset").agg(
        eventos=("event_id", "size"),
        grabaciones=("recording_uid", "nunique"),
        identificados=("identified", "sum"),
        eventos_detector=("use_detector", "sum"),
    )
)


,eventos,grabaciones,identificados,eventos_detector
dataset,,,,
Col-CR,7338,34,6952,7338
Peru,16482,21,14798,16480
PteroSet,15372,563,6702,15371
WABAD-CO,9273,393,9273,9241


## 9. Inventario de audios fuente

In [55]:
raw_by_stem = {}

if RAW_AUDIO_DIR is not None:
    raw_files = []

    for pattern in ["*.wav","*.WAV","*.flac","*.FLAC",]:
        raw_files.extend(RAW_AUDIO_DIR.rglob(pattern))

    for path in raw_files:
        raw_by_stem.setdefault(path.stem,path,)

print("audios fuente indexados:", len(raw_by_stem))


audios fuente indexados: 618



## 10. Construcción de `recordings_manifest`

Se genera una fila por `recording_uid`. Para cada grabación normalizada se registran:

- ruta
- duración
- disponibilidad
- propiedades del archivo fuente cuando están disponibles
- SHA-256 del archivo normalizado
- posibles duplicados binarios


In [56]:
recordings = (master[["recording_uid","dataset","recording","source_group","source_archive","country",]].drop_duplicates("recording_uid").reset_index(drop=True))

target_species_present = (master[master["is_target"]].groupby("recording_uid")["species_scientific"]
    .apply(lambda values: ", " .join(sorted(set(values.dropna())))))

recordings["target_species_present"] = (recordings["recording_uid"].map(target_species_present))

recordings["record_id"] = (recordings["dataset"].map({"PteroSet": 21829388,"Col-CR": 7525349,"Peru": 7079124,"WABAD-CO": 20513304,}))

recordings["normalized_path"] = [str(normalized_audio_path(dataset,recording,))
    for dataset, recording in zip(recordings["dataset"],recordings["recording"],)]

status = []
duration_sec = []
source_sample_rate = []
source_channels = []
normalized_sha256 = []

for _, row in recordings.iterrows():
    normalized_path = Path(row["normalized_path"])

    if not normalized_path.exists():
        status.append("missing")
        duration_sec.append(np.nan)
        source_sample_rate.append(np.nan)
        source_channels.append(np.nan)
        normalized_sha256.append(pd.NA)
        continue

    normalized_info = sf.info(str(normalized_path))

    if normalized_info.samplerate != 32000:
        raise ValueError(f"Sample rate inesperado: {normalized_path}")

    if normalized_info.channels != 1:
        raise ValueError(f"Audio no mono: {normalized_path}")

    duration_sec.append(float(normalized_info.duration))

    normalized_sha256.append(sha256_file(normalized_path))

    raw_path = raw_by_stem.get(Path(row["recording"]).stem)

    if raw_path is not None:
        raw_info = sf.info(str(raw_path))
        source_sample_rate.append(int(raw_info.samplerate))
        source_channels.append(int(raw_info.channels))
        status.append("ok")

    else:
        # propiedades conocidas por la fuente
        if row["dataset"] == "PteroSet":
            source_sample_rate.append(192000)
            source_channels.append(1)

        elif row["dataset"] in ["Col-CR","Peru",]:
            source_sample_rate.append(32000)
            source_channels.append(1)

        else:
            source_sample_rate.append(np.nan)
            source_channels.append(np.nan)

        status.append("already_normalized")

recordings["status"] = status
recordings["duration_sec"] = duration_sec
recordings["source_sample_rate"] = source_sample_rate
recordings["source_channels"] = source_channels
recordings["normalized_sha256"] = normalized_sha256


## 11. Detección de duplicados

In [57]:
recordings["duplicate_of"] = pd.NA
first_recording_by_hash = {}

for index, row in recordings.iterrows():
    sha = row["normalized_sha256"]

    if pd.isna(sha):
        continue

    if sha in first_recording_by_hash:
        recordings.at[index,"duplicate_of",] = first_recording_by_hash[sha]

    else:
        first_recording_by_hash[sha] = row["recording_uid"]

RECORDINGS_COLUMNS = ["recording_uid", "dataset", "recording", "source_group", "source_archive", "country", "target_species_present", "record_id", "status", "normalized_path", "duration_sec", "source_sample_rate", "source_channels", "normalized_sha256", "duplicate_of",]

recordings = recordings[RECORDINGS_COLUMNS]


## 12. Validaciones

In [58]:
# eeructura general
assert len(master) == 48465
assert master["event_id"].is_unique
assert master["recording_uid"].nunique() == 1011
assert len(recordings) == 1011
assert recordings["recording_uid"].is_unique

# cantidad de eventos por fuente
assert len(master[master["dataset"] == "PteroSet"]) == 15372
assert len(master[master["dataset"] == "Col-CR"]) == 7338
assert len(master[master["dataset"] == "Peru"]) == 16482
assert len(master[master["dataset"] == "WABAD-CO"]) == 9273

#isponibilidad de audio
assert master["audio_available"].all()

# Geometria de anotaciones
assert master["use_detector"].sum() == 48430
assert (~master["use_detector"]).sum() == 35

# integridad de archivos normalizados
assert recordings["normalized_sha256"].notna().all()

print("Validaciones completadas correctamente")

display(master.groupby("dataset").agg(eventos=("event_id", "size"), grabaciones=("recording_uid", "nunique"), identificados=("identified", "sum"), detector_validos=("use_detector", "sum"),))
display(recordings.groupby("dataset").agg(grabaciones=("recording_uid", "size"), horas=("duration_sec",lambda values:values.sum() / 3600),))


Validaciones completadas correctamente


,eventos,grabaciones,identificados,detector_validos
dataset,,,,
Col-CR,7338,34,6952,7338
Peru,16482,21,14798,16480
PteroSet,15372,563,6702,15371
WABAD-CO,9273,393,9273,9241


,grabaciones,horas
dataset,,
Col-CR,34,33.997249
Peru,21,21.000000
PteroSet,563,73.620556
WABAD-CO,393,6.549966


## 13. Guardado de manifests

In [59]:
master.to_csv(MASTER_PATH,index=False,)
recordings.to_csv(RECORDINGS_PATH,index=False,)

print("archivos generados:")
print(MASTER_PATH)
print(RECORDINGS_PATH)


archivos generados:
/content/drive/MyDrive/DATASET_PROYECTO/metadata/master_manifest.csv
/content/drive/MyDrive/DATASET_PROYECTO/metadata/recordings_manifest.csv
